In [ ]:
from src_gregor.regression_tree import DecisionTreeRegressor as TreeCustom
from src_gregor.random_forest import RandomForestRegressor as RFCustom

import pandas as pd
import time

# sklearn models, metrics and helpers
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, median_absolute_error

# typing
from typing import Dict, Tuple

def split_df(df: pd.DataFrame, 
             target_name: str, 
             test_size: float = 0.2, 
             random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    '''
    Helpfer function for splitting the df into X and y and subsequently training and test data
    
    :param df: DataFrame
    :type df: pd.DataFrame
    :param target_name: Target column name
    :type target_name: str
    :param test_size: Define the split size in range 0-1
    :type test_size: float
    :param random_state: random state integer
    :type random_state: int
    :return: X_train, X_test, y_train, y_test
    :rtype: Tuple[DataFrame, DataFrame, Series[Any], Series[Any]]
    '''
    X = df.drop(columns=[target_name])
    y = df[target_name]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def evaluate_run(name: str, y_true, y_pred) -> Dict[str, float]:
    '''
    :param name: string name of the model being used
    :param y_true: actual target
    :param y_pred: predicted target 
    :return: dictionary containing standard evaluation metrics from sklearn.metrics
    :rtype: Dict[str, float]
    '''
    return {
        "name": name,
        "r2": r2_score(y_true, y_pred),
        "mse": mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "medianAE" : median_absolute_error(y_true, y_pred)
    }

# ---------- Comparison Runner ----------
def run_comparison(df: pd.DataFrame, 
                   target_col: str, 
                   tree_hyperparameters: dict = {"classifier__max_depth": [3, 5, 7]}, 
                   RF_hyperparameters: dict = {"classifier__max_depth": [3, 5, 7]},
                   GBR_hyperparameters: dict = {"classifier__max_depth": [3, 5]},
                   scoring_technique_cv: str = "r2",
                   preprocessor: object = None,
                   n_estimators: int = 10,
                   cv_folds: int = 5) -> pd.DataFrame:
    '''
    Running/training and comparing our custom implementation vs sklearns 
    regression tree/forest as well as their GradientBoostingRegressor for an additional regression method 
    
    :param df: DataFrame containing features and the target
    :type df: pd.DataFrame
    :param target_col: target column name
    :type target_col: str
    :param *_hyperparameters: hyperparameters to be passed to the respective model in the gridsearchcv
    :type tree_hyperparameters: dict
    :param scoring_technique_cv: scoring technique for gridsearchcv
    :type scoring_technique_cv: str
    :param preprocessor: sklearn transformer (like standardscaler) or just None
    :type preprocessor: object
    :param n_estimators: Number of trees to be trained for the forests
    :type n_estimators: int
    :return: Dataframe containing regression scores (based on evaluate_run())
    :rtype: DataFrame
    '''
    X_train, X_test, y_train, y_test = split_df(df, target_name=target_col)

    # models for comparison
    models = {
        # Trees
        "TreeCustom": (TreeCustom(), tree_hyperparameters),
        "SklearnTree": (DecisionTreeRegressor(), tree_hyperparameters),

        # Random Forests, 1/3 for custom to align with sklearns way of sampling
        "RFCustom": (RFCustom(n_estimators=n_estimators, max_features=1/3), RF_hyperparameters),
        "SklearnRF": (RandomForestRegressor(n_estimators=n_estimators), RF_hyperparameters),

        # Additional Regressor
        "SklearnGB": (GradientBoostingRegressor(), GBR_hyperparameters)
    }

    # scores from the cv in gridsearchcv
    scoring = {
        "r2": "r2",
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "mape": "neg_mean_absolute_percentage_error",
        "medianAE": "neg_median_absolute_error"
    }

    results = []

    for name, (model, param_grid) in models.items():
        pipeline_steps = []
        if preprocessor is not None:
            pipeline_steps.append(("preprocessor", preprocessor))
        pipeline_steps.append(("classifier", model))

        pipe = Pipeline(pipeline_steps)

        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring=scoring,
            refit=scoring_technique_cv, # this is what the hyperparameter will be tuning on
            cv=cv_folds,
            refit=True,
            return_train_score=False
        )

        start_train = time.time()
        grid.fit(X_train, y_train)
        end_train = time.time()
        training_time = end_train - start_train

        best_pipe = grid.best_estimator_

        start_pred = time.time()
        y_pred = best_pipe.predict(X_test)
        end_pred = time.time()
        prediction_time = end_pred - start_pred

        res = evaluate_run(name, y_test, y_pred)

        # CV metrics (mean across folds for best params)
        best_idx = grid.best_index_
        res["cv_r2"] = grid.cv_results_["mean_test_r2"][best_idx]
        res["cv_mse"] = -grid.cv_results_["mean_test_mse"][best_idx]
        res["cv_mae"] = -grid.cv_results_["mean_test_mae"][best_idx]
        res["cv_mape"] = -grid.cv_results_["mean_test_mape"][best_idx]
        res["cv_medianAE"] = -grid.cv_results_["mean_test_medianAE"][best_idx]

        res["best_params"] = grid.best_params_
        res["preprocessor"] = preprocessor.__class__.__name__ if preprocessor is not None else "None"
        res["training_time_sec"] = training_time
        res["prediction_time_sec"] = prediction_time
        
        results.append(res)

    return pd.DataFrame(results)

# laod the data
df_corn = pd.read_csv("data/corn_data_preprocessed.csv")
df_steel  = pd.read_csv("data/steel_industry_data_preprocessed.csv")
df_wave = pd.read_csv("data/wave_energy_farm_preprocessed.csv")

# set the hyperparameters
tree_hyperparameters = {"classifier__max_depth": [3, 5, 7, None],
                       "classifier__min_samples_split": [2, 3, 5, 7],
                       "classifier__min_samples_leaf": [1, 3, 5, 7]}
RF_hyperparameters = {"classifier__max_depth": [3, 5, 7, None],
                       "classifier__min_samples_split": [2, 3, 5, 7],
                       "classifier__min_samples_leaf": [1, 3, 5, 7]}
GBR_hyperparameters = {"classifier__max_depth": [3, 5, 7]}
   

# run the comparisons
results_corn = run_comparison(df=df_corn,
                              target_col="Yield",
                              tree_hyperparameters=tree_hyperparameters,
                              RF_hyperparameters=RF_hyperparameters,
                              GBR_hyperparameters=GBR_hyperparameters)
results_steel = run_comparison(df=df_steel,
                               target_col="Usage_kWh",
                               tree_hyperparameters=tree_hyperparameters,
                               RF_hyperparameters=RF_hyperparameters,
                               GBR_hyperparameters=GBR_hyperparameters)
results_wave = run_comparison(df= df_wave, 
                              target_col="Total_Power",
                              tree_hyperparameters=tree_hyperparameters,
                              RF_hyperparameters=RF_hyperparameters,
                              GBR_hyperparameters=GBR_hyperparameters)

# save the results
results_corn.to_csv("Exercise2/results/corn_results.csv", index=False)
results_steel.to_csv("Exercise2/results/steel_results.csv", index=False)
results_wave.to_csv("Exercise2/results/wave_results.csv", index=False)

# et voila, print em'
print(results_corn)
print(results_steel)
print(results_wave)


In [ ]:
results_corn

In [ ]:
results_steel

In [ ]:
results_wave